# MicroAGI v2 — Nine Pillars, Honestly Measured

Same nine pillars, same environment. Two differences, and they are the whole point.

**1. Four components were provably no-ops. They are fixed.**

| Component | Was | Now |
|---|---|---|
| TTT adapter | `mse(x + f(x), x)` → reduces to `mean(f(x)²)`, drives the adapter to **zero** | self-supervised next-latent prediction — a real task signal |
| Sleep consolidation | `mse(q_proj(x), k_proj(x))` → collapses query and key into the **same function** | replay: retrieved value must match the stored value |
| Value head | never trained; MCTS searched over **random numbers** | TD(0) bootstrapping, so search has signal |
| JEPA | same-encoder stop-grad, collapse-prone | EMA target encoder + variance term, and collapse is **tested for** |

**2. The benchmark can fail.**

The previous version wrote `PASS` into an f-string. It printed nine PASSes on an untrained network. Here every pillar has a **control** and a **threshold**, and prints FAIL when it misses. Some pillars are expected to fail on a short run — that is the harness working, not the harness broken.

Read the number and the control, never the word.

**What "9/9" would mean:** nine modules measurably do their job in a 10×10 gridworld. That is a working cognitive architecture. It is not AGI, and no run of this notebook will be. The pillars are the axes AGI would need, not a checklist that completes it.

## 1 · Setup

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, math, copy, random
from typing import Dict, List, Tuple, Any, Optional, Callable

SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")
print("This notebook runs fine on CPU; the model is ~3.5M params.")

## 2 · Environment (unchanged — this part was already correct)

In [ ]:
class MicroWorld:
    ACTION_NAMES = ["UP","DOWN","LEFT","RIGHT","INTERACT","USE_TOOL"]
    NUM_ACTIONS = 6

    def __init__(self, grid_size=10, view_radius=2, task_mode="standard"):
        self.grid_size=grid_size; self.view_radius=view_radius
        self.task_mode=task_mode; self.max_steps=100; self.current_step=0
        self.grid=np.zeros((grid_size,grid_size),dtype=np.int32)
        self.reset()

    def reset(self, task_mode=None):
        if task_mode is not None: self.task_mode=task_mode
        self.current_step=0; self.grid.fill(0)
        self.has_key=False; self.has_tool=False
        self.terminal_solved=False; self.door_open=False
        g=self.grid_size
        self.grid[0,:]=1; self.grid[-1,:]=1; self.grid[:,0]=1; self.grid[:,-1]=1
        mid=g//2
        self.grid[:,mid]=1
        self.door_pos=[mid,mid]; self.grid[mid,mid]=3
        self.agent_pos=[1,1]
        self.key_pos=[1,mid-2]; self.grid[1,mid-2]=2
        self.terminal_pos=[mid-2,1]; self.grid[mid-2,1]=5
        self.goal_pos=[g-2,g-2]; self.grid[g-2,g-2]=4
        self.hazards=[[mid-1,mid-1],[mid+1,mid+1]]
        for hx,hy in self.hazards: self.grid[hx,hy]=6
        return self._get_obs()

    def _get_obs(self):
        r=self.view_radius; w=2*r+1
        window=np.ones((w,w),dtype=np.float32)
        ax,ay=self.agent_pos
        for i in range(-r,r+1):
            for j in range(-r,r+1):
                gx,gy=ax+i,ay+j
                if 0<=gx<self.grid_size and 0<=gy<self.grid_size:
                    window[i+r,j+r]=float(self.grid[gx,gy])
        flat=window.flatten()/6.0
        regs=np.array([float(self.has_key),float(self.has_tool),
                       float(self.terminal_solved),float(self.door_open),
                       ax/self.grid_size, ay/self.grid_size,
                       self.current_step/self.max_steps],dtype=np.float32)
        obs=np.concatenate([flat,regs])
        return {"obs_vector":torch.from_numpy(obs).float().unsqueeze(0),
                "agent_pos":tuple(self.agent_pos),"has_key":self.has_key,
                "terminal_solved":self.terminal_solved,"door_open":self.door_open,
                "step":self.current_step}

    def step(self, action, tool_output=None):
        self.current_step+=1; reward=-0.01; done=False; info={"event":None}
        ax,ay=self.agent_pos
        moves = ({0:(1,0),1:(-1,0),2:(0,1),3:(0,-1)} if self.task_mode=="shifted_rules"
                 else {0:(-1,0),1:(1,0),2:(0,-1),3:(0,1)})
        if action in moves:
            dx,dy=moves[action]; nx,ny=ax+dx,ay+dy
            cell=self.grid[nx,ny]
            if cell==1: pass
            elif cell==3 and not self.door_open: pass
            elif cell==6:
                reward-=1.0; info["event"]="hit_hazard"; self.agent_pos=[nx,ny]
            else: self.agent_pos=[nx,ny]
        elif action==4:
            for ox,oy in [(-1,0),(1,0),(0,-1),(0,1),(0,0)]:
                tx,ty=ax+ox,ay+oy
                if not (0<=tx<self.grid_size and 0<=ty<self.grid_size): continue
                cell=self.grid[tx,ty]
                if cell==2:
                    self.has_key=True; self.grid[tx,ty]=0
                    reward+=1.0; info["event"]="picked_key"
                elif cell==3:
                    if self.task_mode=="tool_required":
                        if self.terminal_solved:
                            self.door_open=True; self.grid[tx,ty]=0
                            reward+=2.0; info["event"]="door_unlocked_via_tool"
                    elif self.has_key:
                        self.door_open=True; self.grid[tx,ty]=0
                        reward+=2.0; info["event"]="door_unlocked"
        elif action==5:
            if tool_output in (147,"147"):
                self.terminal_solved=True; reward+=1.5; info["event"]="terminal_solved"
        if self.agent_pos==self.goal_pos:
            reward+=10.0; done=True; info["event"]="goal_reached"
        if self.current_step>=self.max_steps: done=True
        return self._get_obs(), reward, done, info

OBS_DIM=32
print("[ok] MicroWorld")

## 3 · Modules — with the four no-ops fixed

Each fix is marked `# FIX:` with what it replaced.

In [ ]:
class JEPAEncoder(nn.Module):
    def __init__(self, obs_dim=OBS_DIM, latent_dim=256, hidden=512):
        super().__init__()
        self.inp=nn.Linear(obs_dim,hidden)
        self.b1=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),nn.Linear(hidden,hidden))
        self.b2=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),nn.Linear(hidden,hidden))
        self.out=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),
                               nn.Linear(hidden,latent_dim),nn.LayerNorm(latent_dim))
    def forward(self,o):
        h=self.inp(o); h=h+self.b1(h); h=h+self.b2(h); return self.out(h)


class TitansMemory(nn.Module):
    def __init__(self, latent_dim=256, memory_dim=128, momentum=0.95):
        super().__init__()
        self.memory_dim=memory_dim; self.momentum=momentum
        self.q_proj=nn.Linear(latent_dim,memory_dim,bias=False)
        self.k_proj=nn.Linear(latent_dim,memory_dim,bias=False)
        self.v_proj=nn.Linear(latent_dim,memory_dim,bias=False)
        self.out_proj=nn.Linear(memory_dim,latent_dim)
        self.gate=nn.Sequential(nn.Linear(latent_dim*2,latent_dim),nn.Sigmoid())
        self.register_buffer("M",torch.zeros(memory_dim,memory_dim))
        self.traces=[]
        self.frozen=False          # for the control arm

    def reset_memory(self):
        self.M.zero_(); self.traces.clear()

    def retrieve(self,s):
        q=self.q_proj(s)
        mem=torch.matmul(q,self.M)
        lat=self.out_proj(mem)
        g=self.gate(torch.cat([s,lat],dim=-1))
        return g*s+(1-g)*lat, mem

    def update_test_time(self,s,surprise,lr=0.05):
        if self.frozen: return
        with torch.no_grad():
            k=self.k_proj(s); v=self.v_proj(s)
            self.M.mul_(self.momentum).add_(torch.matmul(k.t(),v),
                                            alpha=lr*float(surprise)/(self.memory_dim**0.5))
            if len(self.traces)<1000:
                self.traces.append({"latent":s.detach().cpu(),"surprise":surprise})

    def consolidate_sleep(self,optimizer):
        """FIX: was mse(q_proj(x), k_proj(x)), which drives q_proj and k_proj to be
        the SAME function and destroys the key/query distinction the memory needs.
        Now: replay stored traces and require that what we RETRIEVE matches what we
        STORED -- an objective that is zero only when the memory actually works."""
        if len(self.traces)<8: return 0.0
        tr=sorted(self.traces,key=lambda t:t["surprise"],reverse=True)[:32]
        lat=torch.cat([t["latent"] for t in tr],dim=0).to(self.M.device)
        q=self.q_proj(lat)
        retrieved=torch.matmul(q,self.M)
        target=self.v_proj(lat).detach()
        loss=F.mse_loss(retrieved,target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        return float(loss)


class LatentWorldModel(nn.Module):
    def __init__(self, latent_dim=256, num_actions=6, hidden=512):
        super().__init__()
        self.action_embed=nn.Embedding(num_actions,64)
        self.tin=nn.Linear(latent_dim+64,hidden)
        self.r1=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),nn.Linear(hidden,hidden))
        self.r2=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),nn.Linear(hidden,hidden))
        self.tout=nn.Sequential(nn.LayerNorm(hidden),nn.GELU(),
                                nn.Linear(hidden,latent_dim),nn.LayerNorm(latent_dim))
        self.reward_head=nn.Sequential(nn.Linear(latent_dim,128),nn.GELU(),nn.Linear(128,1))
        self.value_head =nn.Sequential(nn.Linear(latent_dim,128),nn.GELU(),nn.Linear(128,1))
    def forward(self,s,a):
        if a.dtype!=torch.long: a=a.long()
        ae=self.action_embed(a)
        if ae.dim()==1: ae=ae.unsqueeze(0)
        h=self.tin(torch.cat([s,ae],dim=-1)); h=h+self.r1(h); h=h+self.r2(h)
        ns=s+self.tout(h)
        return ns, self.reward_head(ns), self.value_head(ns)


class MetacognitiveArbiter(nn.Module):
    def __init__(self, latent_dim=256, p=0.25):
        super().__init__()
        self.probe=nn.Sequential(nn.Linear(latent_dim,256),nn.GELU(),nn.Dropout(p),
                                 nn.Linear(256,128),nn.GELU(),nn.Dropout(p),
                                 nn.Linear(128,2))
    def assess(self,s,n=5):
        was=self.training; self.train()
        preds=torch.stack([self.probe(s) for _ in range(n)],0)
        if not was: self.eval()
        unc=float(preds.var(0).sum())
        conf=float(torch.sigmoid(preds.mean(0)[...,0]))
        if unc>0.08 or conf<0.4:   mode,budget,ttt="System_2_Deep",24,True
        elif unc>0.03:             mode,budget,ttt="System_2_Shallow",8,False
        else:                      mode,budget,ttt="System_1",4,False
        return {"confidence":conf,"epistemic_uncertainty":unc,
                "mode":mode,"search_budget":budget,"trigger_ttt":ttt}


class ModularRouters(nn.Module):
    def __init__(self, latent_dim=256, num_experts=4):
        super().__init__()
        self.router=nn.Linear(latent_dim,num_experts)
        self.experts=nn.ModuleList([
            nn.Sequential(nn.Linear(latent_dim,128),nn.GELU(),nn.Linear(128,latent_dim))
            for _ in range(num_experts)])
    def forward(self,s):
        w=F.softmax(self.router(s),dim=-1)
        outs=torch.stack([e(s) for e in self.experts],dim=-2)
        mixed=(w.unsqueeze(-1)*outs).sum(-2)
        return s+mixed, w


class ReasoningEngine(nn.Module):
    def __init__(self, world_model, num_actions=6, latent_dim=256):
        super().__init__()
        object.__setattr__(self,"world_model",world_model)
        self.num_actions=num_actions
        self.ttt=nn.Sequential(nn.Linear(latent_dim,128),nn.GELU(),nn.Linear(128,latent_dim))

    def adapt_test_time(self,s,obs_next_latent=None,action=None,num_steps=5,lr=0.01):
        """FIX: was mse(s + ttt(s), s), which expands to mean(ttt(s)^2) and drives the
        adapter to zero -- the harder it tried, the less it adapted. Now the adapter is
        trained on a REAL self-supervised signal: make the world model's prediction of
        the next latent match what actually happened. Falls back to a no-op (returning s
        unchanged) when no transition is available, instead of pretending to adapt."""
        if obs_next_latent is None or action is None:
            return s
        params=list(self.ttt.parameters())
        opt=torch.optim.SGD(params,lr=lr)
        tgt=obs_next_latent.detach()
        for _ in range(num_steps):
            opt.zero_grad()
            adapted=s.detach()+self.ttt(s.detach())
            pred,_,_=self.world_model(adapted,action)
            loss=F.mse_loss(pred,tgt)
            loss.backward(); opt.step()
        with torch.no_grad():
            return s+self.ttt(s)

    def plan_mcts(self,s,search_budget=16,gamma=0.95):
        """Depth-2 search over the learned model. Scores are reward + gamma*value, and
        the value head is now TRAINED (see the training loop), so this searches over
        signal rather than over random initialisation."""
        if search_budget<=0:
            return random.randrange(self.num_actions)
        best,best_score=0,-1e9
        with torch.no_grad():
            for a in range(self.num_actions):
                at=torch.tensor([a],device=s.device,dtype=torch.long)
                ns,r,v=self.world_model(s,at)
                score=float(r)+gamma*float(v)
                if search_budget>=8:
                    sub=-1e9
                    for a2 in range(self.num_actions):
                        a2t=torch.tensor([a2],device=s.device,dtype=torch.long)
                        _,r2,v2=self.world_model(ns,a2t)
                        sub=max(sub,float(r2)+gamma*float(v2))
                    score=float(r)+gamma*sub
                if score>best_score: best_score,best=score,a
        return best


class SandboxedToolEngine:
    def __init__(self):
        self.registry={}; self.enabled=True
        self.register("arithmetic_solver",lambda nums:sum(nums))
    def register(self,name,fn): self.registry[name]=fn
    def synthesize_and_register(self,name,code_str,test_input,expected):
        env={"math":math}
        try:
            exec(code_str,env)
            if name not in env: return False
            if env[name](test_input)==expected:
                self.register(name,env[name]); return True
            return False
        except Exception: return False
    def execute(self,name,*a,**k):
        if not self.enabled or name not in self.registry: return None
        try: return self.registry[name](*a,**k)
        except Exception as e: return f"ExecutionError: {e}"

print("[ok] modules (4 fixes applied)")

## 4 · Agent

In [ ]:
class MicroAGI(nn.Module):
    def __init__(self, obs_dim=OBS_DIM, latent_dim=256, num_actions=6, num_experts=4):
        super().__init__()
        self.encoder=JEPAEncoder(obs_dim,latent_dim)
        self.target_encoder=copy.deepcopy(self.encoder)   # FIX: EMA target, anti-collapse
        for p in self.target_encoder.parameters(): p.requires_grad_(False)
        self.memory=TitansMemory(latent_dim)
        self.world_model=LatentWorldModel(latent_dim,num_actions)
        self.metacognition=MetacognitiveArbiter(latent_dim)
        self.adapters=ModularRouters(latent_dim,num_experts)
        self.reasoner=ReasoningEngine(self.world_model,num_actions,latent_dim)
        self.tools=SandboxedToolEngine()
        self.num_actions=num_actions
        self.subgoal=None; self.subgoal_steps=0; self.checkpoints=[]

    @torch.no_grad()
    def ema_update(self,tau=0.99):
        for p,q in zip(self.target_encoder.parameters(),self.encoder.parameters()):
            p.mul_(tau).add_(q,alpha=1-tau)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def reset_episode(self):
        self.memory.reset_memory(); self.subgoal=None
        self.subgoal_steps=0; self.checkpoints.clear()

    def step(self, obs, use_mcts=True):
        o=obs["obs_vector"].to(next(self.parameters()).device)
        with torch.no_grad():
            s_raw=self.encoder(o)
            s_mem,_=self.memory.retrieve(s_raw)
            s_ad,w=self.adapters(s_mem)
            meta=self.metacognition.assess(s_ad)
        self.subgoal_steps+=1
        if self.subgoal is None or self.subgoal_steps>15:
            self.subgoal_steps=0
            if not obs.get("has_key",False) and not obs.get("terminal_solved",False):
                self.subgoal="ACQUIRE_KEY"
            elif not obs.get("door_open",False): self.subgoal="UNLOCK_DOOR"
            else: self.subgoal="REACH_GOAL"
        self.checkpoints.append({"subgoal":self.subgoal})
        a = (self.reasoner.plan_mcts(s_ad,meta["search_budget"]) if use_mcts
             else random.randrange(self.num_actions))
        payload=self.tools.execute("arithmetic_solver",[42,17,88]) if a==5 else None
        self.memory.update_test_time(s_raw,surprise=meta["epistemic_uncertainty"])
        return {"action":a,"tool_payload":payload,"latent":s_raw,
                "uncertainty":meta["epistemic_uncertainty"],
                "budget":meta["search_budget"],"mode":meta["mode"],
                "subgoal":self.subgoal,"routing":w}

model=MicroAGI().to(device)
print(f"[ok] MicroAGI  params = {model.count_parameters():,}")

## 5 · Training

Adds what was missing: **TD(0) value learning**, so the value head MCTS depends on
is no longer random. Also EMA target encoder for JEPA and an explicit variance term,
because a stop-grad against your own encoder can collapse to a constant.

In [ ]:
def train(model, episodes=120, max_steps=40, gamma=0.95, log_every=20, verbose=True):
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    env=MicroWorld()
    hist={"reward":[],"loss":[],"consolidation":[]}
    for ep in range(1,episodes+1):
        model.reset_episode()
        mode="tool_required" if ep%3==0 else "standard"
        obs=env.reset(task_mode=mode)
        ep_r=0.0; ep_l=0.0; n=0
        for _ in range(max_steps):
            eps=max(0.1,1.0-ep/ (episodes*0.6))
            explore=random.random()<eps
            d=model.step(obs,use_mcts=not explore)
            nobs,r,done,info=env.step(d["action"],d["tool_payload"])
            ep_r+=r; n+=1

            o_c=obs["obs_vector"].to(device); o_n=nobs["obs_vector"].to(device)
            at=torch.tensor([d["action"]],device=device,dtype=torch.long)
            rt=torch.tensor([[r]],device=device,dtype=torch.float32)

            s_t=model.encoder(o_c)
            with torch.no_grad():
                s_n_tgt=model.target_encoder(o_n)          # EMA target
            s_n_pred,r_pred,v_pred=model.world_model(s_t,at)

            # JEPA: cosine to EMA target
            jepa=2.0-2.0*(F.normalize(s_n_pred,dim=-1)*F.normalize(s_n_tgt,dim=-1)).sum(-1).mean()
            # anti-collapse: penalise vanishing feature variance
            var_term=F.relu(1.0-s_t.std(dim=0).mean()) if s_t.shape[0]>1 else torch.zeros((),device=device)
            # reward model
            rew=F.mse_loss(r_pred,rt)
            # FIX: TD(0) value target -- the value head was NEVER trained before
            with torch.no_grad():
                _,_,v_next=model.world_model(s_n_tgt,at)
                v_target=rt+(0.0 if done else gamma)*v_next
            val=F.mse_loss(v_pred,v_target)

            loss=jepa+rew+val+0.1*var_term
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); model.ema_update()

            ep_l+=float(loss); obs=nobs
            if done: break
        c=model.memory.consolidate_sleep(opt) if ep%5==0 else 0.0
        hist["reward"].append(ep_r); hist["loss"].append(ep_l/max(1,n))
        hist["consolidation"].append(c)
        if verbose and (ep%log_every==0 or ep==1):
            print(f"  ep {ep:3d}/{episodes} | {mode:13s} | steps {n:2d} | "
                  f"reward {ep_r:+7.2f} | loss {ep_l/max(1,n):.4f}")
    return hist

print("training...")
hist=train(model, episodes=120)
print("[ok] trained")

## 6 · The benchmark

Every test: a **control**, a **threshold**, and a real verdict. `assert_pass` is the
only place a verdict is produced, and it compares numbers.

In [ ]:
RESULTS=[]
def record(pillar, metric, value, control, threshold, rule, note=""):
    passed = rule(value, control, threshold)
    RESULTS.append({"pillar":pillar,"metric":metric,"value":value,
                    "control":control,"threshold":threshold,
                    "pass":bool(passed),"note":note})
    return passed

GT = lambda v,c,t: v > c + t          # must beat control by margin
LT = lambda v,c,t: v < c - t          # must undercut control by margin

def rollout(model, env, task_mode="standard", steps=40, use_mcts=True, tools=True):
    model.reset_episode(); model.tools.enabled=tools
    obs=env.reset(task_mode=task_mode); total=0.0; reached=False; events=set()
    for _ in range(steps):
        d=model.step(obs,use_mcts=use_mcts)
        obs,r,done,info=env.step(d["action"],d["tool_payload"])
        total+=r
        if info["event"]: events.add(info["event"])
        if done:
            reached = info["event"]=="goal_reached"; break
    model.tools.enabled=True
    return total, reached, events

env=MicroWorld()
LD=256

# ---------------- P1 Durable Memory ----------------
# store A, write 30 distractors, then check we retrieve A and not the distractors.
def memory_recall(frozen):
    model.memory.reset_memory(); model.memory.frozen=frozen
    A=model.encoder(torch.randn(1,OBS_DIM,device=device)).detach()
    model.memory.update_test_time(A,surprise=1.0,lr=0.2)
    for _ in range(30):
        model.memory.update_test_time(
            model.encoder(torch.randn(1,OBS_DIM,device=device)).detach(),
            surprise=0.05,lr=0.02)
    _,got=model.memory.retrieve(A)
    want=model.memory.v_proj(A)
    model.memory.frozen=False
    return float(F.cosine_similarity(got,want,dim=-1))

live=memory_recall(False); ctl=memory_recall(True)
record("P1 Durable Memory","cos(retrieved, stored)",live,ctl,0.10,GT,
       "control = memory frozen (must be ~0)")

# ---------------- P2 Latent Reasoning ----------------
# planning must beat random action selection.
mcts_r=[rollout(model,env,"standard",40,use_mcts=True)[0]  for _ in range(15)]
rand_r=[rollout(model,env,"standard",40,use_mcts=False)[0] for _ in range(15)]
record("P2 Latent Reasoning","mean episode reward (MCTS)",
       float(np.mean(mcts_r)),float(np.mean(rand_r)),0.5,GT,
       f"control = random policy; sd {np.std(mcts_r):.2f} vs {np.std(rand_r):.2f}")

# ---------------- P3 OOD / TTT ----------------
# on inverted-rule task, adaptation must REDUCE next-latent prediction error.
env.reset(task_mode="shifted_rules")
errs_before=[]; errs_after=[]
obs=env.reset(task_mode="shifted_rules")
for _ in range(20):
    a=random.randrange(6)
    at=torch.tensor([a],device=device,dtype=torch.long)
    o_c=obs["obs_vector"].to(device)
    nobs,_,done,_=env.step(a)
    o_n=nobs["obs_vector"].to(device)
    with torch.no_grad():
        s=model.encoder(o_c); s_n=model.encoder(o_n)
        p,_,_=model.world_model(s,at)
        errs_before.append(float(F.mse_loss(p,s_n)))
    s_ad=model.reasoner.adapt_test_time(s,obs_next_latent=s_n,action=at,num_steps=5)
    with torch.no_grad():
        p2,_,_=model.world_model(s_ad,at)
        errs_after.append(float(F.mse_loss(p2,s_n)))
    obs=nobs
    if done: obs=env.reset(task_mode="shifted_rules")
record("P3 OOD Generalization","prediction error AFTER adaptation",
       float(np.mean(errs_after)),float(np.mean(errs_before)),0.0,LT,
       "control = same error before adaptation")

# ---------------- P4 Sample Efficiency ----------------
# JEPA-pretrained encoder vs fresh encoder: episodes to reach reward threshold.
def episodes_to_threshold(m, thresh=-0.5, cap=40):
    e=MicroWorld()
    for i in range(1,cap+1):
        r,_,_=rollout(m,e,"standard",40,use_mcts=True)
        if r>=thresh: return i
    return cap
fresh=MicroAGI().to(device)
ep_trained=episodes_to_threshold(model)
ep_fresh=episodes_to_threshold(fresh)
record("P4 Sample Efficiency","episodes to reach reward threshold",
       float(ep_fresh),float(ep_trained),0.0,GT,
       "PASS = trained reaches it sooner than an untrained net")
# collapse check
with torch.no_grad():
    batch=torch.randn(64,OBS_DIM,device=device)
    feat_std=float(model.encoder(batch).std(0).mean())
record("P4b Representation health","mean feature std (collapse check)",
       feat_std,0.0,0.05,GT,"PASS = representation has NOT collapsed to a constant")

# ---------------- P5 Long-Horizon Agency ----------------
# must complete key -> door -> goal, not just wander.
def subgoal_progress(use_mcts):
    hits=0
    for _ in range(15):
        _,_,ev=rollout(model,env,"standard",60,use_mcts=use_mcts)
        score=sum([("picked_key" in ev),("door_unlocked" in ev),("goal_reached" in ev)])
        hits+=score
    return hits/ (15*3)
agent_prog=subgoal_progress(True); rand_prog=subgoal_progress(False)
record("P5 Long-Horizon Agency","subgoal chain completion rate",
       agent_prog,rand_prog,0.05,GT,"control = random policy on same chain")

# ---------------- P6 Task Transfer ----------------
# trained-on-standard model vs untrained, both zero-shot on tool_required.
tr=[rollout(model,env,"tool_required",40)[0] for _ in range(10)]
fr=[rollout(fresh,env,"tool_required",40)[0] for _ in range(10)]
record("P6 Task Transfer","zero-shot reward on unseen task",
       float(np.mean(tr)),float(np.mean(fr)),0.3,GT,
       "control = model that never trained")

# ---------------- P7 World Model ----------------
# THE decisive test: must beat the identity baseline (predict s_{t+1} = s_t).
obs=env.reset(); wm_err=[]; id_err=[]
for _ in range(60):
    a=random.randrange(6); at=torch.tensor([a],device=device,dtype=torch.long)
    o_c=obs["obs_vector"].to(device); nobs,_,done,_=env.step(a)
    o_n=nobs["obs_vector"].to(device)
    with torch.no_grad():
        s=model.encoder(o_c); s_n=model.encoder(o_n)
        p,_,_=model.world_model(s,at)
        wm_err.append(float(F.mse_loss(p,s_n)))
        id_err.append(float(F.mse_loss(s,s_n)))     # identity baseline
    obs=nobs
    if done: obs=env.reset()
record("P7 World Model","next-latent prediction error",
       float(np.mean(wm_err)),float(np.mean(id_err)),0.0,LT,
       "control = predicting no change at all (identity)")

# ---------------- P8 Metacognition ----------------
# uncertainty must CORRELATE with actual error. This is the real test.
obs=env.reset(); us=[]; es=[]
for _ in range(80):
    o_c=obs["obs_vector"].to(device)
    with torch.no_grad():
        s=model.encoder(o_c)
        s_ad,_=model.adapters(*model.memory.retrieve(s)[:1])
        m=model.metacognition.assess(s_ad)
    a=random.randrange(6); at=torch.tensor([a],device=device,dtype=torch.long)
    nobs,_,done,_=env.step(a); o_n=nobs["obs_vector"].to(device)
    with torch.no_grad():
        s_n=model.encoder(o_n); p,_,_=model.world_model(s,at)
        es.append(float(F.mse_loss(p,s_n)))
    us.append(m["epistemic_uncertainty"]); obs=nobs
    if done: obs=env.reset()
def spearman(a,b):
    ra=np.argsort(np.argsort(a)); rb=np.argsort(np.argsort(b))
    return float(np.corrcoef(ra,rb)[0,1])
rho=spearman(us,es)
record("P8 Metacognition","Spearman(uncertainty, actual error)",
       rho,0.0,0.15,GT,"control = zero correlation (uncertainty is noise)")

# ---------------- P9 Tool Use ----------------
# synthesis must verify, AND the tool must actually change task outcome.
syn=model.tools.synthesize_and_register(
    "poly","def poly(x): return x**3 - 2*x + 5", test_input=2, expected=9)
with_tool=[rollout(model,env,"tool_required",50,tools=True)[0]  for _ in range(12)]
without  =[rollout(model,env,"tool_required",50,tools=False)[0] for _ in range(12)]
record("P9 Tool Use","reward on tool-gated task (tool ON)",
       float(np.mean(with_tool)),float(np.mean(without)),0.2,GT,
       f"control = identical runs with tools disabled; synthesis verified={syn}")

print("benchmark complete")

## 7 · Results

In [ ]:
print("="*104)
print(f"{'PILLAR':<28}{'METRIC':<38}{'VALUE':>10}{'CONTROL':>10}{'MARGIN':>8}{'VERDICT':>9}")
print("="*104)
n_pass=0
for r in RESULTS:
    v="PASS" if r["pass"] else "FAIL"
    n_pass+=r["pass"]
    print(f"{r['pillar']:<28}{r['metric']:<38}{r['value']:>10.4f}"
          f"{r['control']:>10.4f}{r['threshold']:>8.2f}{v:>9}")
print("="*104)
print(f"  {n_pass}/{len(RESULTS)} pillars pass against their controls")
print("="*104)
print()
for r in RESULTS:
    if r["note"]: print(f"  {r['pillar']:<28} {r['note']}")
print()
print("How to read this:")
print("  - A FAIL is information, not a defect. It says that module does not yet")
print("    beat its control, which is the thing the old notebook could never tell you.")
print("  - Train longer (episodes=400) and re-run; watch which verdicts flip.")
print("  - P7 is the one to watch: beating the identity baseline is the difference")
print("    between a world model and a very expensive copy operation.")

## 8 · What a full pass would and would not mean

If all nine flip to PASS, you have nine modules that measurably beat their controls in
a 10×10 gridworld. That is a real, working cognitive architecture and a legitimate thing
to have built.

It is not AGI, and running it longer will not make it AGI. The gap is not effort — it is
that these are nine *axes*, and gridworld competence on each does not compose into general
intelligence. Every axis here is solved at toy scale in a fully observable world with six
actions and one goal. The open problem is each of them at open-ended scale, and no one has
solved that.

What you can honestly claim from a full pass: *"I implemented nine cognitive mechanisms and
verified each against a control that could fail."* Most published work on this doesn't
include the control. That claim is worth more than a fake nine-of-nine.